In [584]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

from torchinfo import summary

In [308]:
# keep user reviews and item reviews separately??

import pandas as pd

df = pd.read_csv("../data/books_mini_sample.csv")

In [314]:
filtered_df = df[['rating', 'text', 'user_id', 'asin']].copy()
filtered_df.head()

,rating,text,user_id,asin
0,5.0,This is a great book with a lot of detail. ON...,AE224GVO7OHTYF26U6ER6BEVIUAQ,0486241386
1,5.0,Ladies 'n Gentlemen...we have another hit from...,AE225Z2VRWT6GPTOMA4H4O3H2KVQ,B000MGATTE
2,5.0,I ordered this book for my Kindle. Koontz and ...,AE225Z2VRWT6GPTOMA4H4O3H2KVQ,B000UDNBRQ
3,5.0,Absolutely loved this book! Stephen King just...,AE225Z2VRWT6GPTOMA4H4O3H2KVQ,B000UZJREU
4,5.0,Another King satisfied reader~! what can I sa...,AE225Z2VRWT6GPTOMA4H4O3H2KVQ,B001RF3U9K


In [315]:
filtered_df.dtypes

rating     float64
text        object
user_id     object
asin        object
dtype: object

In [316]:
filtered_df['user_id'] = filtered_df['user_id'].astype('category').cat.codes
filtered_df['asin'] = filtered_df['asin'].astype('category').cat.codes

filtered_df.head()

,rating,text,user_id,asin
0,5.0,This is a great book with a lot of detail. ON...,0,1655
1,5.0,Ladies 'n Gentlemen...we have another hit from...,1,5766
2,5.0,I ordered this book for my Kindle. Koontz and ...,1,5817
3,5.0,Absolutely loved this book! Stephen King just...,1,5819
4,5.0,Another King satisfied reader~! what can I sa...,1,5877


In [242]:
import gensim.downloader as api

word2vec_model = api.load('word2vec-google-news-300')

# Access word embeddings from the pre-trained model
vector = word2vec_model['word']

In [ ]:
import numpy as np
word2vec_model['<pad>'] = np.zeros(300)

In [317]:
%%time
max_length = 100
filtered_df['text'] = filtered_df['text'].astype('string')\
                                        .fillna('')\
                                        .str.split(' ')\
                                        .apply(lambda words: [word for word in words if word in word2vec_model.key_to_index])

filtered_df['text'] = filtered_df['text'].apply(
    lambda x: x[:max_length] + ['<pad>'] * (max_length - len(x)) if len(x) < max_length else x[:max_length]
)

CPU times: user 1.01 s, sys: 215 ms, total: 1.23 s
Wall time: 1.3 s


In [318]:
filtered_df['text']

0         [This, is, great, book, with, lot, ONE, CAUTIO...
1         [Ladies, have, another, hit, from, Stephen, th...
2         [I, ordered, this, book, for, my, Koontz, two,...
3         [Absolutely, loved, this, Stephen, King, just,...
4         [Another, King, satisfied, what, can, I, The, ...
                                ...                        
105975    [Funny, book, daughter, did, book, report, on,...
105976    [Lot, great, information, on, Use, it, almost,...
105977    [Love, this, book, <pad>, <pad>, <pad>, <pad>,...
105978    [7, year, old, loves, this, workbook, we, go, ...
105979    [we, all, him, he, chosen, Its, potter, I, lov...
Name: text, Length: 105980, dtype: object

In [324]:
%%time
from torch.utils.data import Dataset, DataLoader


class RecSysDataset(Dataset):
    def __init__(self, df, word2vec_model, max_reviews):
        self.df = df
        self.word2vec_model = word2vec_model
        self.max_reviews = max_reviews

    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        user_reviews = self.df[self.df['user_id'] == row.user_id]['text']
        w2v_user_tensor = [torch.from_numpy(self.word2vec_model[i]) for i in user_reviews if len(i)!=0]
        # TODO - need to remove the padding as we already appending <pad> token and standardizing the review length
        user_tower_input = pad_sequence(w2v_user_tensor, batch_first=True, padding_value=0)

        item_reviews = self.df[self.df['asin'] == row.asin]['text']
        w2v_item_tensor = [torch.from_numpy(self.word2vec_model[i]) for i in item_reviews if len(i)!=0]
        # TODO - need to remove the padding as we already appending <pad> token and standardizing the review length
        item_tower_input = pad_sequence(w2v_item_tensor, batch_first=True, padding_value=0)

        target_size = (self.max_reviews, max_length, 300)
        user_tower_input = self._resize_tensors(user_tower_input, target_size)
        item_tower_input = self._resize_tensors(item_tower_input, target_size)

        return user_tower_input, item_tower_input, row['rating'], row['user_id'], row['asin']

    def _resize_tensors(self, input_tensor, target_size):

        # Resize the num_reviews if needed; TODO - change to random sampling? picking only first `max_reviews` number of reviews of the user or item
        tensor_size = input_tensor.shape
        if tensor_size[0] != target_size[0]:
            if tensor_size[0] < target_size[0]:
                diff = target_size[0] - tensor_size[0]
                padding = torch.zeros((diff, tensor_size[1], tensor_size[2]))
                output_tensor = torch.cat((input_tensor, padding), dim=0)
            else:
                output_tensor = input_tensor[:target_size[0]]
        else:
            output_tensor = input_tensor
        return output_tensor



CPU times: user 96 μs, sys: 258 μs, total: 354 μs
Wall time: 251 μs


In [325]:
%%time
# num_reviews, review_length, word_dim
# num_reviews, review_length, word_dim

# do we need to use negative sampling??
# padding a tensor?
# remove any words that are not present in the word2vec vocabulary

recsys_dataset = RecSysDataset(filtered_df, word2vec_model, 10)
dataloader = DataLoader(recsys_dataset, batch_size=32, shuffle=True)

for i, sample in enumerate(dataloader):
    if i > 2:
        break
    user_tower_input, item_tower_input, label, user_id, asin = sample
    print(user_tower_input.shape, item_tower_input.shape)
    

torch.Size([32, 10, 100, 300]) torch.Size([32, 10, 100, 300])
torch.Size([32, 10, 100, 300]) torch.Size([32, 10, 100, 300])
torch.Size([32, 10, 100, 300]) torch.Size([32, 10, 100, 300])
CPU times: user 4.17 s, sys: 901 ms, total: 5.07 s
Wall time: 1.4 s


In [579]:
class NRCMA(nn.Module):
    def __init__(self, 
                 num_users, 
                 num_products,
                 num_conv_filters,
                 word_embed_dim,
                 id_embed_dim,
                 attention_vector_dim,
                 feature_vector_dim
                 ):
        super(NRCMA, self).__init__()
        self.user_embedding = nn.Embedding(num_users, id_embed_dim)
        self.item_embedding = nn.Embedding(num_products, id_embed_dim)

        # user tower
        self.user_cnn = nn.Conv2d(in_channels=1, out_channels=num_conv_filters, kernel_size=(1, word_embed_dim), padding=0, bias=True)
        self.word_level_matrix = nn.Linear(id_embed_dim, attention_vector_dim)
        self.word_harmony_matrix = nn.Linear(num_conv_filters, attention_vector_dim)
        self.review_level_matrix = nn.Linear(id_embed_dim, attention_vector_dim)
        self.review_harmony_matrix = nn.Linear(num_conv_filters, attention_vector_dim)

        # item tower
        self.item_cnn = nn.Conv2d(in_channels=1, out_channels=num_conv_filters, kernel_size=(1, word_embed_dim), padding=0, bias=True)
        self.item_word_level_matrix = nn.Linear(id_embed_dim, attention_vector_dim)
        self.item_word_harmony_matrix = nn.Linear(num_conv_filters, attention_vector_dim)
        self.item_review_level_matrix = nn.Linear(id_embed_dim, attention_vector_dim)
        self.item_review_harmony_matrix = nn.Linear(num_conv_filters, attention_vector_dim)

        # factorization machine, 2 * num_conv_filters as we get concat features from user, item
        self.fm_linear = nn.Linear(2*num_conv_filters, 1)
        self.v = nn.Parameter(torch.empty(2*num_conv_filters, feature_vector_dim))
        
        # Initialize self.v similar to nn.Linear weights
        nn.init.kaiming_uniform_(self.v, a=math.sqrt(5))


    def forward(self, user_input, item_input, user_id, item_id):
        
        user_embedding = self.user_embedding(user_id)
        item_embedding = self.item_embedding(item_id)

        d_u = self.process_single_tower(user_input, item_embedding, tower='user')
        d_i = self.process_single_tower(item_input, user_embedding, tower='item')
        o = torch.cat((d_u, d_i), dim=1)

        # factorization machine
        prediction = self.fm_linear(o).item() + (o @ torch.triu(self.v @ self.v.T, diagonal=1) @ o.T).item()
        return prediction

    def process_single_tower(self, input_matrix, embedding, tower):

        if tower == 'user':
            cnn = self.user_cnn
            word_level_matrix = self.word_level_matrix
            word_harmony_matrix = self.word_harmony_matrix
            review_level_matrix = self.review_level_matrix
            review_harmony_matrix = self.review_harmony_matrix
        else:
            cnn = self.item_cnn
            word_level_matrix = self.item_word_level_matrix
            word_harmony_matrix = self.item_word_harmony_matrix
            review_level_matrix = self.item_review_level_matrix
            review_harmony_matrix = self.item_review_harmony_matrix

        # process information through single tower - employing cross attention
        review_features = F.relu(cnn(input_matrix))
        beta_k = F.relu(word_level_matrix(embedding))
        
        intermediate_output_1 = word_harmony_matrix(review_features.squeeze().transpose(-1, -2)).transpose(-1, -2) # not sure if this step is correct; need to check this
        print(beta_k.shape, intermediate_output_1.shape)
        b_c = beta_k @ intermediate_output_1
        alpha_c = nn.Softmax(dim=2)(b_c)
        
        d_uk = review_features.squeeze() @ torch.transpose(alpha_c, -1, -2)

        beta_u = F.relu(review_level_matrix(embedding))
        intermediate_output_2 = review_harmony_matrix(d_uk.squeeze())
        print(beta_u.shape, intermediate_output_2.shape)
        b_k = beta_u @ intermediate_output_2.T

        alpha_k = nn.Softmax(dim=1)(b_k)
        print(alpha_k.shape, d_uk.shape)
        d_u = alpha_k @ d_uk.squeeze()

        return d_u


In [580]:
for i, sample in enumerate(recsys_dataset):
    if i > 2:
        break
    user_tower_input, item_tower_input, label, user_id, asin = sample
    print(user_tower_input.shape, item_tower_input.shape)

torch.Size([10, 100, 300]) torch.Size([10, 100, 300])
torch.Size([10, 100, 300]) torch.Size([10, 100, 300])
torch.Size([10, 100, 300]) torch.Size([10, 100, 300])


In [582]:
model = NRCMA(num_users=16398, 
             num_products=7889,
             num_conv_filters=80,
             word_embed_dim=300,
             id_embed_dim=30,
             attention_vector_dim=20,
             feature_vector_dim=10)

In [588]:
summary(model)

Layer (type:depth-idx)                   Param #
NRCMA                                    1,600
├─Embedding: 1-1                         491,940
├─Embedding: 1-2                         236,670
├─Conv2d: 1-3                            24,080
├─Linear: 1-4                            620
├─Linear: 1-5                            1,620
├─Linear: 1-6                            620
├─Linear: 1-7                            1,620
├─Conv2d: 1-8                            24,080
├─Linear: 1-9                            620
├─Linear: 1-10                           1,620
├─Linear: 1-11                           620
├─Linear: 1-12                           1,620
├─Linear: 1-13                           161
Total params: 787,491
Trainable params: 787,491
Non-trainable params: 0

In [583]:
user_tower_input = user_tower_input.unsqueeze(1)
item_tower_input = item_tower_input.unsqueeze(1)
model(user_tower_input, item_tower_input, torch.tensor([user_id], dtype=torch.int), torch.tensor([asin], dtype=torch.int))

torch.Size([1, 20]) torch.Size([10, 20, 100])
torch.Size([1, 20]) torch.Size([10, 20])
torch.Size([1, 10]) torch.Size([10, 80, 1])
torch.Size([1, 20]) torch.Size([10, 20, 100])
torch.Size([1, 20]) torch.Size([10, 20])
torch.Size([1, 10]) torch.Size([10, 80, 1])


0.054031722247600555